In [ ]:
%matplotlib inline

import numpy as np
import scanpy as sc
import torch
import pandas as pd
import scipy
import sys
sys.path.append("/home/anirudhn/Krushna/Thymus/Trajectory-Pos/Margaret/Margaret-main/margaret")
random_seed = 0
np.random.seed(random_seed)
torch.manual_seed(random_seed)
import matplotlib.pyplot as plt
rc_parms = {"figure.figsize": [4, 4], "figure.dpi": 300, "font.size": 10, "font.family": "Arial"}
save_parms = {"bbox_inches": "tight", "transparent": True}

In [ ]:
# Load data
data_path = "/home/anirudhn/Krushna/Thymus/Data/Thymus_CITE-seq-v1.0.0/YosefLab-Thymus_CITE-seq-9c20db9/totalVI_AllData/posterior_adata.h5ad"
adata_all = sc.read_h5ad(data_path)
adata_all

In [ ]:
filtered_df = pd.DataFrame(data=adata_all.obsm['denoised_genes'], index=adata_all.obs_names, columns=adata_all.var_names)
data = sc.AnnData(filtered_df)
data.obs = adata_all.obs[["batch","annotations"]]
data.obsm['X_totalVI'] = adata_all.obsm['X_totalVI']
data.obsm['X_umap'] = adata_all.obsm['X_umap']

In [ ]:
with plt.rc_context(rc_parms):
    ax = sc.pl.umap(data, color=["annotations"], frameon= False, return_fig=True, title='')
    plt.savefig("./Results/Figures/All-emb.png", **save_parms)


In [ ]:
pos_cells = pd.read_csv("../Results/Pos_subset.csv", header=None)
pos_cells = pos_cells[0].values

In [ ]:
data = data[pos_cells,:].copy()
data.obs['annotations'] = data.obs['annotations'].astype(str)

In [ ]:
del data.uns['annotations_colors']

In [ ]:
print(data)
data.obs['annotations'] = data.obs['annotations'].astype('category')
data.obs['annotations'].value_counts()

In [ ]:
Posclusters = ["DP (Sig.)", "Immature CD4", "Immature CD8", "Mature CD4", "Mature CD8"]#, "Interferon sig.", "Neg. sel. (2)", "Treg"]
data = data[data.obs['annotations'].isin(Posclusters),:].copy()
data

In [ ]:
data = sc.read_h5ad("./pos-margaret-0.3-totalvi.h5ad")

In [ ]:
with plt.rc_context(rc_parms):
    sc.pl.umap(data, color="annotations", frameon= False, return_fig=True, title = '')
    plt.savefig("./Results/Figures/Positive-emb.png", **save_parms)


In [ ]:
sc.pp.neighbors(data, use_rep='X_totalVI')

In [ ]:
for res in [0.1,0.2,0.3,0.4,0.5,0.7]:
    print(res)
    sc.tl.leiden(data, resolution=res)
    sc.pl.umap(data, color = ['annotations', 'leiden'])

In [ ]:
import warnings
from train_metric import train_metric_learner

with warnings.catch_warnings():
    # Filter out user warnings from PyTorch about saving scheduler state
    warnings.simplefilter("ignore")
    train_metric_learner(data, n_episodes=5, n_metric_epochs=30, obsm_data_key='X_totalVI', code_size=10,
        backend='leiden', device='cuda', save_path='./Results/Thymus',
        cluster_kwargs={'random_state': 0, 'resolution': 0.3}, nn_kwargs={'random_state': 0, 'n_neighbors': 50},
        trainer_kwargs={'optimizer': 'SGD', 'lr': 0.01, 'batch_size': 256}
    )

In [ ]:
data.uns['metric_clustering_scores'] = list(map(str,data.uns['metric_clustering_scores']))

In [ ]:
import numpy as np
from models.ti.connectivity import compute_directed_cluster_connectivity, compute_undirected_cluster_connectivity
from models.ti.graph import compute_trajectory_graph, compute_connectivity_graph
from utils.plot import plot_connectivity_graph, plot_trajectory_graph
from utils.util import get_start_cell_cluster_id

import networkx as nx

from sklearn.neighbors import NearestNeighbors
from models.ti.pseudotime import compute_pseudotime
from models.ti.pseudotime_v2 import compute_pseudotime
from models.ti.graph import compute_trajectory_graph_v2
from utils.plot import plot_trajectory_graph_v2
from utils.plot import plot_pseudotime

from utils.plot import generate_plot_embeddings, plot_gene_expression, plot_embeddings, plot_clusters
import matplotlib.pyplot as plt



In [ ]:
X_embedded = generate_plot_embeddings(data.obsm['metric_embedding'], method='umap', random_state=random_seed) #preprocessed_data
data.obsm['X_met_embedding'] = X_embedded #preprocessed_data
data.obs['metric_clusters'] = data.obs['metric_clusters'].astype('category')

In [ ]:
data = sc.read_h5ad("pos-margaret-0.3-totalvi.h5ad")

In [ ]:
# data.write("pos-margaret-0.3-totalvi.h5ad")

In [ ]:
with plt.rc_context(rc_parms):
    sc.pl.embedding(data, basis = 'X_met_embedding', color=['annotations'],frameon=False, return_fig=True, title = '')
    plt.savefig("./Results/Figures/margaret-emb.png", **save_parms) 
    sc.pl.embedding(data, basis = 'X_met_embedding', color=['metric_clusters'],frameon=False, return_fig=True, title = '')
    plt.savefig("./Results/Figures/margaret-emb-clus.png", **save_parms)
# plot_embeddings(X_embedded, s=5)#labels

In [ ]:
sc.pl.umap(data, color = ['metric_clusters'], legend_loc='on data')

In [ ]:
data.obs.groupby(['annotations','metric_clusters']).size().unstack().T

In [ ]:
# sc.pp.neighbors(data, use_rep='metric_embedding')
X = data.obsm['metric_embedding']

n_neighbors = 10
nbrs = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean").fit(X)
adj_dist = nbrs.kneighbors_graph(X, mode="distance")
adj_conn = nbrs.kneighbors_graph(X)

In [ ]:
communities = data.obs['metric_clusters'].to_numpy().astype(np.int32)
# adj_conn = data.obsp['connectivities']
# adj_dist = data.obsp['distances']

# start_cell_ids = data[data.obs['annotations_clean'].isin(["DP (Sig.)"]),:].obs_names.to_list()
# start_cluster_ids = get_start_cell_cluster_id(data, start_cell_ids, communities)
start_cluster_ids = {1}
start_cell_ids = data[data.obs['metric_clusters'].isin(list(start_cluster_ids)),:].obs_names.to_list()
# print(type(start_cluster_ids))
print(f'start cell ids {start_cluster_ids}')

In [ ]:
un_connectivity, un_z_score = compute_undirected_cluster_connectivity(communities, adj_conn, z_threshold=1.5)

In [ ]:
plot_connectivity_graph(data.obsm['X_met_embedding'], communities, un_connectivity, mode='undirected', offset=0.2, cmap='Blues', node_size=750)

In [ ]:
connectivity, z_score = compute_directed_cluster_connectivity(communities, adj_conn, threshold=0)

In [ ]:
# v2 pseudotime
G_undirected, node_positions = compute_connectivity_graph(data.obsm['X_met_embedding'], data.obs['metric_clusters'], un_connectivity)
adj_cluster = nx.to_pandas_adjacency(G_undirected)
pseudotime = compute_pseudotime(data, start_cell_ids, adj_dist, adj_cluster)

In [ ]:
with plt.rc_context(rc_parms):
    sc.pl.embedding(data,'X_met_embedding', color = 'metric_pseudotime_v2', cmap='plasma', frameon=False, return_fig=True, title='')
    # plt.savefig("./Results/Figures/Psudotime.png", **save_parms)


In [ ]:
# Compute directed graph v2
plot_trajectory_graph_v2(pseudotime, adj_cluster, data.obs['metric_clusters'], connectivity, node_positions, offset=0.2,node_size=2000, font_size = 20)

In [ ]:
def compute_trajectory_graph_v2(
    pseudotime, adj_cluster, communities, d_connectivity, norm=False
):
    n_communities = np.unique(communities).shape[0]
    cluster_ids = np.unique(communities)

    adj = pd.DataFrame(
        np.zeros((n_communities, n_communities)), index=cluster_ids, columns=cluster_ids
    )

    # Create cluster index
    cluster_pt = pd.DataFrame(index=cluster_ids)
    for idx in cluster_ids:
        cluster_idx = communities == idx
        cluster_pt.loc[idx, "t"] = np.mean(pseudotime.loc[cluster_idx])
    cols = adj_cluster.columns
    for idx in cluster_ids:
        connected_c_idx = cols[adj_cluster.loc[idx, :] != 0]
        for c_idx in connected_c_idx:
            if (cluster_pt.loc[c_idx, "t"] > cluster_pt.loc[idx, "t"]) and (
                adj_cluster.loc[c_idx, idx] != 0
            ):
                # The edge weight will be the contribution from the directed
                # connectivities and difference of the pseudotimes
                adj.loc[idx, c_idx] = d_connectivity.loc[idx, c_idx] + 1 / (
                    1 + np.exp(cluster_pt.loc[c_idx, "t"] - cluster_pt.loc[idx, "t"])
                )

    # Normalize the directed adjacency matrix
    if norm:
        adj = adj.div(adj.sum(axis=1), axis=0).fillna(0)
    return adj

# with cell type pie chats
def plot_trajectory_graph_v3(
    pseudotime,
    adj_cluster,
    communities,
    d_connectivity,
    node_positions,
    adata,
    cell_type = 'annotations',
    start_cell_ids=None,
    cmap="YlGn",
    figsize=(4,4),
    node_size=300,
    font_color="black",
    title=None,
    start_node_color=None,
    node_color=None,
    save_path=None,
    save_kwargs={},
    offset=0,
    **kwargs,
):
    adj_g = compute_trajectory_graph_v2(
        pseudotime, adj_cluster, communities, d_connectivity
    )
    g = nx.from_pandas_adjacency(adj_g, create_using=nx.DiGraph)

    if start_cell_ids is not None:
        start_cell_ids = (
            start_cell_ids if isinstance(start_cell_ids, list) else [start_cell_ids]
        )
    else:
        start_cell_ids = []

    start_cluster_ids = set([communities.loc[id] for id in start_cell_ids])

    colors = np.unique(communities)
    if node_color is not None:
        colors = []
        for c_id in np.unique(communities):
            if c_id in start_cluster_ids and start_node_color is not None:
                colors.append(start_node_color)
            else:
                colors.append(node_color)

    # Draw the graph
    fig = plt.figure(figsize=figsize)
    ax = plt.axes([0,0,1,1])
    
    if title is not None:
        plt.title(title)
    plt.axis("off")
    
    edge_weights = [offset + w for _, _, w in g.edges.data("weight")]
    
    nx.draw_networkx(
        g,
        pos=node_positions,
        cmap=cmap,
        node_color=colors,
        font_color=font_color,
        node_size=node_size,
        width=edge_weights,
        **kwargs,
    )
    
    trans = ax.transData.transform
    trans2 = fig.transFigure.inverted().transform

    piesize = 0.105##node_size*0.027/800#800->0.05 
    p2 = piesize/2.0
    # cs = cm.Set1(np.arange(15)/15.)
    
    for n in g:
        xx,yy = trans(node_positions[n]) # figure coordinates
        xa,ya = trans2((xx,yy)) # axes coordinates
        a = plt.axes([xa-p2,ya-p2, piesize, piesize])
        plt.title(n,**kwargs)
        a.set_aspect('equal')
        
        adata_node = adata[adata.obs['metric_clusters'] == n].copy()
        keys = adata.obs[cell_type].value_counts().keys()
        fracs = []
        colour_list = []
        color_map = adata.uns[cell_type+'_colors']
        for i,key in enumerate(sorted(keys)):
            fracs_keys = adata_node.obs[cell_type].value_counts().keys() 
            colour_list += [color_map[i]]
            if key in fracs_keys:
                fracs += [adata_node.obs[cell_type].value_counts()[key]]
            else:
                fracs += [0]
                
        fracs = np.array(fracs)/adata_node.shape[0] #[15,30,35, 10, 10]
        a.pie(fracs, colors = colour_list) # labels = keys
        
    plt.legend(sorted(keys),bbox_to_anchor = (9,5)) # loc = "lower left", 
    
    if save_path is not None:
        plt.savefig(save_path, **save_kwargs)
with plt.rc_context(rc_parms):
    plot_trajectory_graph_v3(pseudotime, adj_cluster, data.obs['metric_clusters'], connectivity, 
                             node_positions, offset=0.1, adata = data,node_size=500,
                            save_path = "./Results/Figures/Trajectory.png", save_kwargs = save_parms)#, fontsize = 20)


In [ ]:
from models.ti.downstream import (
    get_terminal_states,
    get_terminal_cells,
    sample_waypoints,
    compute_diff_potential
)

In [ ]:
import sys, importlib
importlib.reload(sys.modules['models.ti.downstream'])


In [ ]:
# del get_terminal_states
# del get_terminal_cells
# del sample_waypoints
# del compute_diff_potential

In [ ]:
G_directed_v2 = compute_trajectory_graph_v2(pseudotime, adj_cluster, data.obs['metric_clusters'], connectivity)
terminal_clusters = get_terminal_states(data, G_directed_v2, start_cell_ids)#, mad_multiplier=0.7)
t_cell_ids = get_terminal_cells(data)
_ = sample_waypoints(data, adj_dist.todense(), n_waypoints=30) #dists, wp

In [ ]:
ent, bps = compute_diff_potential(data, adj_dist.todense(), adj_cluster, prune_wp_graph=True) #std_factor=0.6, 

# Plot the Differentiation potential
plot_embeddings(
    data.obsm['X_met_embedding'],
    s=1,
    c=ent,
    figsize=(8, 8),
    cmap='plasma',
    show_colorbar=True,
    cb_axes_pos=[0.92, 0.55, 0.02, 0.3],
    save_kwargs={
        'dpi': 300,
        'bbox_inches': 'tight',
        'transparent': True
    }
)

In [ ]:
ts_map = {
    0: "Mature CD8",
    6: "Mature CD4,"
}
color_map = {
    0: '#9467bd',
    6: '#d62728',
}

In [ ]:
for pro in ["CD24","CD62L","CD55"]:
    match =  adata_all.obs.columns[adata_all.obs.columns.to_series().str.contains(pro)]
    data.obs[pro] = adata_all.obs.loc[data.obs_names, match.values[0]]

In [ ]:
# Plot lineage trends
import importlib
importlib.reload(sys.modules['utils.plot'])
from utils.plot import plot_lineage_trends


comms = data.obs['metric_clusters'].loc[bps.columns]
bps_ = pd.DataFrame(bps.to_numpy(), columns=list(comms), index=data.obs_names)

In [ ]:
with plt.rc_context(rc_parms):
    for gene in ['Rag1', 'Cxcr4', 'Trbc1' , 'Ccr9', 'Ccr4', 'Ccr7', 'H2-K1', 'Klf2', 'S1pr1', "CD24","CD62L","CD55"]:
        plot_lineage_trends(
            data,
            bps_,
            [gene],
            pseudotime_key='metric_pseudotime_v2',
            figsize=(3,3),
            # imputed_key='X_magic',
            nrows=1,
            norm=True,
            ts_map=ts_map,
            save_path='./Results/Figures/Trends/'+gene+'.png',
            save_kwargs={
                'dpi': 300,
                'bbox_inches': 'tight',
                'transparent': True
            },
            color_map=color_map,
            loc = None,
            show_title = False,
            set_ylabel = None
            # threshold=0
        )

In [ ]:
data_raw = data.copy()
data_raw.X = adata_all[data_raw.obs_names,:].X


In [ ]:
for pro in ["CD24","CD62L","CD55"]:
    match =  adata_all.obs.columns[adata_all.obs.columns.to_series().str.contains(pro)]
    data_raw.obs[pro] = adata_all.obs.loc[data_raw.obs_names, match.values[0]]


In [ ]:
importlib.reload(sys.modules['utils.plot'])
from utils.plot import plot_lineage_trends

In [ ]:
for gene in ['Rag1', 'Cxcr4', 'Trbc1' , 'Ccr9', 'Ccr4', 'Ccr7', 'H2-K1', 'Klf2', 'S1pr1', "CD24","CD62L","CD55"]:
    print(gene)
    plot_lineage_trends(
        data_raw,
        bps_,
        [gene],
        pseudotime_key='metric_pseudotime_v2',
        figsize=(3,3),
        # imputed_key='X_magic',
        nrows=1,
        norm=True,
        ts_map=ts_map,
        show_title=True,
        # save_path='./lineage_1.png',
        save_kwargs={
            'dpi': 300,
            'bbox_inches': 'tight',
            'transparent': True
        },
        color_map=color_map,
        
        # threshold=0
    )